In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import os
import re

In [15]:
data = pd.read_csv("adam_sentences_filtered_with_targets.csv")


print(data["ABBR"].unique())

<StringArray>
[ 'AAA',  'ABG',  'ABR',  'ACE',  'ACL',  'ADA',  'ADL',  'AFB',  'AFP',
  'AGA',
 ...
  'TBI',  'TEE', 'TIPS',  'TPA',  'TSH',  'TTP',  'VAC',  'VAD', 'VATS',
 'VBAC']
Length: 136, dtype: str


In [2]:
data = pd.read_csv('adam_dataset_results.csv')
print(len(data))
# get minimum number of variants
print(data["Primary_Num_Variants"].min())
print(data["Primary_Num_Variants"].max())
print(data["Primary_Num_Variants"].mean())
print(data["Primary_Num_Variants"].median())
print(data["Primary_Num_Variants"].std())
print(data["Primary_Num_Variants"].var())
print(data["Alternative_Num_Variants"].min())
print(data["Alternative_Num_Variants"].max())
print(data["Alternative_Num_Variants"].mean())
print(data["Alternative_Num_Variants"].median())
print(data["Alternative_Num_Variants"].std())
print(data["Alternative_Num_Variants"].var())
# Long form scores
print(data["Primary_Long_Form_Score"].min())
print(data["Primary_Long_Form_Score"].max())
print(data["Primary_Long_Form_Score"].mean())   
print(data["Primary_Long_Form_Score"].median())
print(data["Primary_Long_Form_Score"].std())
print(data["Primary_Long_Form_Score"].var())
print(data["Alternative_Long_Form_Score"].min())
print(data["Alternative_Long_Form_Score"].max())
print(data["Alternative_Long_Form_Score"].mean())
print(data["Alternative_Long_Form_Score"].median())
print(data["Alternative_Long_Form_Score"].std())
print(data["Alternative_Long_Form_Score"].var())





137
25
22403
1820.5912408759125
605.0
3156.1977486758087
9961584.228746243
20
2181
174.02919708029196
82.0
274.75662871642476
75491.20502361529
0.5029
0.9954
0.8896109489051095
0.9508
0.12504321587004377
0.01563580583512237
0.204
0.9871
0.8030503649635037
0.8842
0.20225888772883874
0.040908657665307


In [ ]:
data = pd.read_csv('medal-emnlp/full_data.csv')


,TEXT,LOCATION,LABEL
0,alphabisabolol has a primary antipeptic action...,56,substrate
1,a report is given on the recent discovery of o...,24|49|68|113|137|172,carcinosarcoma|recovery|reference|recovery|aft...
2,the virostatic compound nndiethyloxotetradecyl...,55,substrate
3,rmi rmi and rmi are newly synthetized nrdibenz...,25|82|127|182|222,compounds|compounds|inhibitory|lethal doses|ca...
4,a doubleblind study with intraindividual compa...,22|26|28|77|90|144|158|203,oxazepam|placebo|oral administration|pentagast...


In [8]:
data["TEXT"][1]

'a report is given on the recent discovery of outstanding immunological properties in ba ncyanoethyleneurea having a low molecular mass m experiments in ds CS bearing wistar rats have shown that ba at a dosage of only about percent ld mg kg and negligible lethality percent results in a REC rate of percent without hyperglycemia and in one test of percent with hyperglycemia under otherwise unchanged conditions the REF substance ifosfamide if a further development of cyclophosphamide applied without hyperglycemia in its most efficient dosage of percent ld mg kg brought about a recovery rate of percent at a lethality of percent contrary to ba min hyperglycemia caused no further improvement of the REC rate however this comparison is characterized by the fact that both substances exhibit two quite different complementary mechanisms of action leucocyte counts made T3 application of the said cancerostatics and dosages have shown a pronounced stimulation with ba and with ifosfamide the known su

In [12]:
# read ClinicalSenseInventoryII_RefinedMasterFile.txt and create a dataframe separated by |
# clinical_data = pd.read_csv('ClinicalSenseInventoryII_RefinedMasterFile.txt', sep='|')
# clinical_data.head()
clinical_data = pd.read_csv(
    'ClinicalSenseInventoryII_RefinedMasterFile.txt',
    sep='|',
    keep_default_na=False,
    na_values=[''],  # only empty fields are missing
)

In [13]:
clinical_data.head()

,SF,LF,MetaMap CUI,CSI,Ratio in CSI,UMLS CUI,UMLS SOURCE,ADAM,Ratio in ADAM,Dictionary
0,5-FU,"2,4(1H,3H)-Pyrimidinedione, 5-fluoro-;5-Fluoro...",NaN,NaN,NaN,C0016360,MSH;NCI;NDFRT;PDQ,NaN,NaN,NaN
1,5-FU,"2,4-Dioxo-5-fluoropyrimidine",NaN,NaN,NaN,C0016360,NCI;PDQ,NaN,NaN,NaN
2,5-FU,5 Fluorouracil;5 fluorouracil;5-Fluorouracil;5...,C0016360,1.0,1.0,C0016360,CHV;CSP;LCH;LNC;MSH;MTH;MTHSPL;NCI;NDFRT;PDQ;R...,1.0,0.9959,NaN
3,5-FU,5-flourouracil,NaN,NaN,NaN,NaN,NaN,1.0,0.0018,NaN
4,5-FU,FU,C0016360;C1424823,NaN,NaN,C0016360,PDQ,NaN,NaN,NaN


In [19]:
clinical_data["SF"].unique()


sf_list = clinical_data["SF"].unique()
# filter out any that have number such as 5-FU
sf_list = [sf for sf in sf_list if sf.isalpha()]

#save to text file
with open('sf_list.txt', 'w') as f:
    for sf in sf_list:
        f.write(sf + '\n')




In [20]:
adam_results = pd.read_csv('adam_results.csv')
adam_results.head()


,ABBR,Long_Form,Num_Variants,Long_Form_Score,Count
0,AAA,abdominal aortic aneurysm,1215,0.9805,1619
1,ABG,arterial blood gas,99,0.9698,99
2,AB,Alcian blue,79,0.7381,79
3,ABR,Auditory brainstem response,703,0.9474,903
4,ACE,angiotensin-converting enzyme,4868,0.8365,5038


In [25]:
adam_sentences = pd.read_csv('adam_sentences.csv')
adam_sentences.head()

,ABBR,Long_Form,Sentence_Num,PMID,Sentence
0,AAA,abdominal aortic aneurysm,1,16226959,BACKGROUND: One adverse outcome of endovascula...
1,AAA,abdominal aortic aneurysm,2,16171581,OBJECTIVE: abdominal aortic aneurysm ( AAA ) r...
2,AAA,abdominal aortic aneurysm,3,16171580,METHODS: The Canadian Institute for Health Inf...
3,AAA,abdominal aortic aneurysm,4,16134913,Screening for abdominal aortic aneurysm ( AAA ...
4,AAA,abdominal aortic aneurysm,5,16127278,abdominal aortic aneurysm ( AAA ) is character...


In [2]:
dataset = pd.read_excel("triple_sentence_test_set.xlsx")

In [3]:
dataset.head()

,abbr,sentence_a,sentence_b,sentence_c,target
0,AAA,OBJECTIVE: abdominal aortic aneurysm represen...,OBJECTIVE: ( AAA ) represents a chronic degene...,OBJECTIVE: aromatic amino acid represents a ch...,represents
1,AAA,METHODS: The Canadian Institute for Health Inf...,METHODS: The Canadian Institute for Health Inf...,METHODS: The Canadian Institute for Health Inf...,between
2,AAA,Screening for abdominal aortic aneurysm reduc...,Screening for ( AAA ) reduced mortality caused...,Screening for aromatic amino acid reduced mort...,reduced
3,AAA,abdominal aortic aneurysm is characterized by...,( AAA ) is characterized by dilatation of art...,aromatic amino acid is characterized by dilata...,characterized
4,AAA,OBJECTIVE: To evaluate the results of our expe...,OBJECTIVE: To evaluate the results of our expe...,OBJECTIVE: To evaluate the results of our expe...,identify


In [7]:
from datasets import load_dataset
dataset = load_dataset("awinml/medqa")
dataset['train'].to_pandas()
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    validation: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1272
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})

In [10]:
dataset['train'][0]

{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?',
 'answer': 'Nitrofurantoin',
 'options': {'A': 'Ampicillin',
  'B': 'Ceftriaxone',
  'C': 'Doxycycline',
  'D': 'Nitrofurantoin'},
 'meta_info': 'step2&3',
 'answer_idx': 'D',
 'metamap_phrases': ['23 year old pregnant woman',
  'weeks presents',
  'burning',
  'urination',
  'states',
  'started 1 day',
  'worsening',
  'drinking',
  'water',
  'taking cranberry extract',
  'feels well',
  'follo

In [8]:
data = pd.read_csv("adam_dataset_patching.csv")
data.head()


,ABBR,Sentence_A,Sentence_B,Sentence_C,target,Primary_Expression,Alternative_Expression
0,AAA,OBJECTIVE: abdominal aortic aneurysm represent...,OBJECTIVE: AAA represents a chronic degenerati...,OBJECTIVE: aromatic amino acids represents a c...,represents,NaN,NaN
1,AAA,METHODS: The Canadian Institute for Health Inf...,METHODS: The Canadian Institute for Health Inf...,METHODS: The Canadian Institute for Health Inf...,between,NaN,NaN
2,AAA,Screening for abdominal aortic aneurysm reduce...,Screening for AAA reduced mortality caused by ...,Screening for aromatic amino acids reduced mor...,reduced,NaN,NaN
3,AAA,abdominal aortic aneurysm is characterized by ...,AAA is characterized by dilatation of arterial...,aromatic amino acids is characterized by dilat...,is,NaN,NaN
4,AAA,OBJECTIVE: To evaluate the results of our expe...,OBJECTIVE: To evaluate the results of our expe...,OBJECTIVE: To evaluate the results of our expe...,identify,NaN,NaN


In [7]:
# add empty column name "primary expression"
data["Primary_Expression"] = ""
data["Alternative_Expression"] = ""
data.head()
# save as csv
data.to_csv("adam_dataset_patching.csv", index=False)
